<a href="https://colab.research.google.com/github/nawroz-m/ML_learning/blob/main/gan_faces_partial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os, sys
import time

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import sklearn.datasets

import torch
import torch.autograd as autograd
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from torchvision import transforms
import pandas as pd
from PIL import Image
from datetime import datetime
from torch.utils.tensorboard import SummaryWriter
from torchsummary import summary


In [4]:
# define hyperparameters

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f'Your are working on [{DEVICE}] device')
DIM = 64 # Model dimensionality
BATCH_SIZE = 7**2 # Batch size
CRITIC_ITERS = 5 # For WGAN and WGAN-GP, number of critic iters per gen iter
LAMBDA = 5 # Gradient penalty lambda hyperparameter
LEARNING_RATE = 1e-4
BETAS = (0.5, 0.9)
ITERS = 20000000 # How many generator iterations to train for

Your are working on [cpu] device


In [5]:
# define dataset

class Dataset(torch.utils.data.Dataset):

    def __init__(self, root_dir, transform=None):
        # read the csv file
        self.fns = [fn for fn in os.listdir(root_dir) if fn.endswith('.jpg')]
        self.root_dir = root_dir
        self.transform = transform


    def __len__(self):
        # here i will return the number of samples in the dataset
        return len(self.fns)


    def __getitem__(self, idx):
        # get filename
        cur_sample = self.fns[idx]
        fn = os.path.join(self.root_dir, cur_sample)
        # open image with PIL
        image = Image.open(fn).convert('RGB')
        # reshape to 64x64
        image = image.resize((64, 64), Image.BILINEAR)
        # convert to numpy
        image = np.array(image)
        # apply transform
        if self.transform:
            image = self.transform(image)
        # convert to torch format
        image = transforms.ToTensor()(image)
        # return values
        return image

In [6]:
# connect to drive
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip "/content/drive/MyDrive/Big Img Sig/GANs & Diffusion Models/img_align_celeba.zip"

In [97]:
# convert in dataloader
dl = torch.utils.data.DataLoader(
    Dataset(root_dir='img_align_celeba'),
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    num_workers=8
    )
len(dl)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


4134

In [98]:
# convert in iterator

def inf_train_gen(dl):
    while True:
        for batch in dl:
            yield batch

data_iter = inf_train_gen(dl)

In [99]:
# test dataloader
boh = next(data_iter)
print(boh.shape)

torch.Size([49, 3, 64, 64])


In [55]:
class Upsampling_block(nn.Module):
  def __init__(self, in_ch, out_ch):
      super().__init__()
      self.up=nn.ConvTranspose2d(in_ch, out_ch, 4, 2, 1)
      self.relu = nn.ReLU()
  def forward(self, x):
    return self.relu(self.up(x))

upsample = Upsampling_block(1, 64).to(DEVICE)
summary(upsample, input_size=(1, 64, 64))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
   ConvTranspose2d-1         [-1, 64, 128, 128]           1,088
              ReLU-2         [-1, 64, 128, 128]               0
Total params: 1,088
Trainable params: 1,088
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.02
Forward/backward pass size (MB): 16.00
Params size (MB): 0.00
Estimated Total Size (MB): 16.02
----------------------------------------------------------------


In [88]:
# Define generator

class Generator(nn.Module):
    def __init__(self, DIM=64):
        super(Generator, self).__init__()
        self.DIM = DIM
        # TODO:
        # create a Sequential containing a linear layer and a relu. The linear layer
        # that takes a vector of size 128 and outputs a vector of size (4*4)*(4*DIM),
        # which will be the number of pixels of the first 4x4 activation map that will
        # have 4*DIM channels. The relu will be applied after the linear layer.
        # Note that in forward, we will reshape the output of this sequential to be
        # of size (B, 4*DIM, 4, 4).
        self.preprocess = nn.Sequential(
            nn.Linear(64, (4*4)*(8*self.DIM)), # [49, 4096]
            nn.ReLU(True),
        )

        # TODO: create four blocks of upsampling. Each block will be a Sequential
        # containing a ConvTranspose2d and a ReLU (except the last). The ConvTranspose2d
        # will take as input the number of channels of the previous activation map and
        # will output half the number of channels. The kernel size will be 4, the stride
        # will be 2, and the padding will be 1.

        self.up1 = Upsampling_block(8*self.DIM, 4*self.DIM)
        self.up2 = Upsampling_block(4*self.DIM, 2*self.DIM)
        self.up3 = Upsampling_block(2*self.DIM, self.DIM)
        self.up4 = nn.ConvTranspose2d(self.DIM, 3, 4, 2, 1)

        # TODO: define a sigmoid layer to make outputs between 0 and 1
        self.output = nn.Sigmoid()


    def forward(self, input):
        # TODO: apply the layers
        x = self.preprocess(input)
        x = x.view(x.shape[0], 8*self.DIM, 4, 4) # [49, 512, 4, 4]
        up1 = self.up1(x)
        up2 = self.up2(up1)
        up3 = self.up3(up2)
        up4 = self.up4(up3)
        output = self.output(up4)
        return output

In [89]:
# test generator

netG = Generator(3)
netG.to(DEVICE)
noise = torch.randn(BATCH_SIZE, 64).to(DEVICE)
print(f'noise shape: {noise.shape}')
fake = netG(noise)
print(f'fake shape: {fake.shape}')

noise shape: torch.Size([49, 64])
fake shape: torch.Size([49, 3, 64, 64])


In [74]:
summary(netG, input_size=(64,))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                  [-1, 384]          24,960
              ReLU-2                  [-1, 384]               0
   ConvTranspose2d-3             [-1, 12, 8, 8]           4,620
              ReLU-4             [-1, 12, 8, 8]               0
  Upsampling_block-5             [-1, 12, 8, 8]               0
   ConvTranspose2d-6            [-1, 6, 16, 16]           1,158
              ReLU-7            [-1, 6, 16, 16]               0
  Upsampling_block-8            [-1, 6, 16, 16]               0
   ConvTranspose2d-9            [-1, 3, 32, 32]             291
             ReLU-10            [-1, 3, 32, 32]               0
 Upsampling_block-11            [-1, 3, 32, 32]               0
  ConvTranspose2d-12            [-1, 1, 64, 64]              49
          Sigmoid-13            [-1, 1, 64, 64]               0
Total params: 31,078
Trainable params: 

In [90]:
class Conv_leakyRelu_block(nn.Module):
  def __init__(self, in_ch, out_ch):
      super().__init__()
      self.net = nn.Sequential(
        nn.Conv2d(in_ch, out_ch, 4, 2, 1),
        nn.LeakyReLU())
  def forward(self, x):
    return self.net(x)

leaky_rb = Conv_leakyRelu_block(1, 64)
summary(leaky_rb, input_size=(1, 64, 64))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 64, 32, 32]           1,088
         LeakyReLU-2           [-1, 64, 32, 32]               0
Total params: 1,088
Trainable params: 1,088
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.02
Forward/backward pass size (MB): 1.00
Params size (MB): 0.00
Estimated Total Size (MB): 1.02
----------------------------------------------------------------


In [91]:
# Define discriminator

class Discriminator(nn.Module):
    def __init__(self, DIM=64):
        super(Discriminator, self).__init__()
        self.DIM = DIM

        # TODO: define a sequence of conv-leakyrelu layers that will reduce the size of the
        # input from 64x64 to 4x4. The number of channels will be doubled at each layer.
        # From DIM to 8*DIM.
        self.conv1 =Conv_leakyRelu_block(3, self.DIM)
        self.conv2 =Conv_leakyRelu_block(self.DIM, 2*self.DIM)
        self.conv3 =Conv_leakyRelu_block(2*self.DIM, 4*self.DIM)
        self.conv4 =Conv_leakyRelu_block(4*self.DIM, 8*self.DIM)

        # TODO: create a linear layer that will take as input the number of pixels of the
        # last activation map and will output a vector of size 1.
        self.fc = nn.Linear(8*self.DIM * 4 * 4, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, input):
        # TODO: apply all layers
        x = self.conv1(input)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = x.view(x.size(0), -1)

        x = self.fc(x)
        out = 0
        out = self.sigmoid(x)
        return out

dis = Discriminator(3)
summary(dis, input_size=(3, 64, 64))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1            [-1, 3, 32, 32]             147
         LeakyReLU-2            [-1, 3, 32, 32]               0
Conv_leakyRelu_block-3            [-1, 3, 32, 32]               0
            Conv2d-4            [-1, 6, 16, 16]             294
         LeakyReLU-5            [-1, 6, 16, 16]               0
Conv_leakyRelu_block-6            [-1, 6, 16, 16]               0
            Conv2d-7             [-1, 12, 8, 8]           1,164
         LeakyReLU-8             [-1, 12, 8, 8]               0
Conv_leakyRelu_block-9             [-1, 12, 8, 8]               0
           Conv2d-10             [-1, 24, 4, 4]           4,632
        LeakyReLU-11             [-1, 24, 4, 4]               0
Conv_leakyRelu_block-12             [-1, 24, 4, 4]               0
           Linear-13                    [-1, 1]             385
          Sigmoid-14          

In [92]:
# test discriminator

netD = Discriminator(3)
netD.to(DEVICE)
fake = torch.rand(BATCH_SIZE, 3, 64, 64)
fake = fake.to(DEVICE)
print(f'fake: {fake.shape}')
output = netD(fake)
print(output.shape)



fake: torch.Size([49, 3, 64, 64])
torch.Size([49, 1])


In [93]:
# Define function for gradient penalty

def calc_gradient_penalty(netD, real_data, fake_data, LAMBDA=LAMBDA, device='cuda'):
    # detach
    real_data = real_data.detach()
    fake_data = fake_data.detach()
    #print real_data.size()
    alpha = torch.rand(BATCH_SIZE, 1, 1, 1)
    alpha = alpha.expand_as(real_data)
    alpha = alpha.to(device)

    # alpha blending between real and fake
    interpolates = alpha * real_data + ((1 - alpha) * fake_data)
    interpolates = interpolates.to(device)
    interpolates.requires_grad = True
    # let discriminator evaluate interpolates
    disc_interpolates = netD(interpolates)
    # compute gradients
    gradients = autograd.grad(outputs=disc_interpolates, inputs=interpolates,
                              grad_outputs=torch.ones(disc_interpolates.size()).to(device),
                              create_graph=True, retain_graph=True, only_inputs=True)[0]
    # compute penalty
    gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean() * LAMBDA
    return gradient_penalty



In [ ]:
# define training routine

# TODO: define the two networks, the two optimizers. Move the nets in the DEVICE.
# Note that optimizers also have betas.

# 1. define two networks
netG = Generator(3)
netG = netG.to(DEVICE)
netD = Discriminator(3)
netD = netD.to(DEVICE)
# 2. define two optimizers
optimizerG = torch.optim.Adam(netG.parameters(), lr=0.0002, betas=(0.0, 0.9))
optimizerD = torch.optim.Adam(netD.parameters(), lr=0.0002, betas=(0.0, 0.9))

# get the current date and time
dt = datetime.now()
# create summary writer
writer = SummaryWriter(f'experiments/{dt}')



for iteration in range(ITERS):

    ############################
    # (1) Update D network
    ###########################

    # enable gradient computation for discriminator
    for p in netD.parameters():
        p.requires_grad = True

    # train discriminator for CRITIC_ITERS iterations
    for iter_d in range(CRITIC_ITERS):
        # reset gradients
        # netD.zero_grad()
        optimizerD.zero_grad()

        # get real data
        try:
          real_data = next(data_iter)
        except StopIteration:
          data_iter = iter(dl)
          real_data = next(data_iter)
        real_data = real_data.to(DEVICE)

        # NOTE:
        # the discriminator loss is: D(fake) - D(real) + gradient_penalty
        # to avoid excessive memory consumption we compute the three
        # terms separately. This is possible as the derivative of a sum
        # is the sum of the derivatives.

        # TODO: train with real: -D(real)
        # 1 - forward pass
        # 2 - average the output of the discriminator
        # 3 - invert the sign of the average
        # 4 - backward pass
        # 1- forward pass
        D_real = netD(real_data)
        # 2- average the output of the discriminator
        D_real = D_real.mean()
        # 3- invert the sign of the average
        # 4- backward pass
        (-D_real).backward()

        # TODO: create synthetic images
        # 1 - create noise
        # 2 - without gradient computation, generate fake images

        # 1- create noise
        noise = torch.randn(BATCH_SIZE, 64).to(DEVICE)
        # 2- Without gradient computation, generate the fake images
        fake = netG(noise).detach()

        # TODO: train with fake: D(fake)
        # same thing as before, but without inverting the sign
        output_fake = netD(fake)
        D_fake = output_fake.mean()
        D_fake.backward()
        # TODO: compute gradient penalty
        # use the function
        gradient_penalty = calc_gradient_penalty(netD=netD,
                                                 real_data=real_data,
                                                 fake_data=fake,
                                                 LAMBDA=LAMBDA,
                                                 device=DEVICE)
        gradient_penalty.backward()
        # compute wasserstain distance
        Wasserstein_D = D_real - D_fake

        # update discriminator weights
        optimizerD.step()

        # plot
        if iter_d == 0:
            writer.add_scalar('D_real', D_real, iteration)
            writer.add_scalar('D_fake', D_fake, iteration)
            writer.add_scalar('gradient_penalty', gradient_penalty, iteration)
            writer.add_scalar('Wasserstein_D', Wasserstein_D, iteration)


    ############################
    # (2) Update G network
    ###########################
    # stop gradient computation on discriminator's weights
    # (we are not updating the discriminator but just using it)
    for p in netD.parameters():
        p.requires_grad = False  # to avoid computation

    # reset gradients
    # netG.zero_grad()
    # netD.zero_grad()
    optimizerG.zero_grad()


    # NOTE: the loss of the generator is -D(G(fake))

    # TODO: create synthetic sample - G(fake)
    noise = torch.randn(BATCH_SIZE, 64).to(DEVICE)
    fake = netG(noise)

    # TODO: evaluate it with discriminator: -D(G(fake))
    G = -netD(fake).mean()
    G.backward()
    # update generator weights
    optimizerG.step()

    # plot
    writer.add_scalar('G', G, iteration)

    # view images
    if iteration % 100 == 0:
        writer.add_images('real', real_data, iteration)
        writer.add_images('fake', fake, iteration)

    # save parameters
    if iteration % 1000 == 0:
        torch.save({
            'netG': netG.state_dict(),
            'netD': netD.state_dict(),
            'optimizerG': optimizerG.state_dict(),
            'optimizerD': optimizerD.state_dict(),
        }, f'experiments/{dt}/model.pth')